# Duration Core MVP

Provisional historical inspection of the accepted Duration Core definition. These seven-level scores are ordinal economic judgments, not return estimates or ETF rankings.

Run the calculation section to explicitly acquire, calculate, and save all three exposure histories. After restarting the kernel, the visualization section can run on its own using only the saved CSV and JSON. See `README.md` for commands and cache behavior.

## Calculation — explicit acquisition through persistence

In [11]:
from pathlib import Path
from duration_config import load_config
from fred_data import acquire_fred
from duration_components import calculate_components
from duration_core import evaluate_core
from duration_results import save_results

BASE = Path.cwd()
config = load_config(BASE / "duration_config.yaml")
REFRESH_FRED = False  # Set True for an explicit fresh download; False reuses valid caches.

In [12]:
# 1. Acquire Raw Observations; the local raw cache is separate from Core results.
observations, source_snapshot = acquire_fred(BASE / "cache", refresh=REFRESH_FRED)
print(f"Loaded {len(observations):,} source observations across all three Treasury tenors.")

Loaded 41,619 source observations across all three Treasury tenors.


In [3]:
# 2. Calculate tenor-local Features, Component Values, and Component States.
components = calculate_components(observations, config["trend"], config["recent_move"])
print("Components calculated for all tenors; components retains M5 and endpoint lineage.")

Components calculated for all tenors; components retains M5 and endpoint lineage.


In [5]:
components.tail()

,date,raw_yield_pct,series_id,yield_pct,segment,availability,observation_index,smoothed_yield_bp,trend_reference_bp,recent_reference_bp,trend_value_bp,recent_move_value_bp,trend_reference_date,recent_reference_date,trend_state,recent_move_state
41614,2026-10-02,5.63,DGS30,5.63,2,accepted,5166,560.6,489.8,524.0,70.8,36.6,2026-04-03,2026-09-02,Rising,Rising
41615,2026-10-05,5.66,DGS30,5.66,2,accepted,5167,562.6,489.4,525.2,73.2,37.4,2026-04-06,2026-09-03,Rising,Rising
41616,2026-10-06,5.64,DGS30,5.64,2,accepted,5168,563.6,489.8,525.6,73.8,38.0,2026-04-07,2026-09-04,Rising,Rising
41617,2026-10-07,5.67,DGS30,5.67,2,accepted,5169,564.2,489.4,525.6,74.8,38.6,2026-04-08,2026-09-08,Rising,Rising
41618,2026-10-08,5.60,DGS30,5.60,2,accepted,5170,564.0,489.8,525.8,74.2,38.2,2026-04-09,2026-09-09,Rising,Rising


In [6]:
# 3. Map already calculated Components to all three synthetic Duration Exposures.
core_results = evaluate_core(components, config["exposures"], config["rules"])
print(f"Completed {core_results['core_score'].notna().sum():,} Core Duration judgments.")

Completed 38,348 Core Duration judgments.


In [7]:
# 4. Persist completed results and provenance before using visualization controls.
saved_csv, saved_metadata = save_results(core_results, source_snapshot, config, BASE / "output")
print(f"Saved {saved_csv.name} and {saved_metadata.name}.")

Saved duration_core.csv and duration_core_metadata.json.


## Visualization — runnable alone after a kernel restart

Run every cell below without executing the calculation section. This loads the saved snapshot; subsequent YAML edits do not change it. The saved metadata supplies the historical-event presets. To require the current analytical configuration to match, explicitly call `load_results(..., expected_config=load_config(...))`; a mismatch raises and requires a new calculation/save.

The default Custom period uses shared available dates. You can extend the dates to inspect an exposure's earlier history. Event presets preserve previous Custom dates. Gaps and warmup observations are unavailable, and no lines bridge structural discontinuities.

In [1]:
from pathlib import Path
from IPython.display import display
from duration_results import load_results
from duration_plot import DurationControls

# 5. Load persisted analytical results independently of acquisition/calculation.
loaded_results, saved_metadata = load_results(Path.cwd() / "output")
print(f"Saved snapshot generated at {saved_metadata['generated_at_utc']}.")
print(f"Calculation fingerprint: {saved_metadata['calculation_fingerprint']}")

Saved snapshot generated at 2026-10-12T03:56:57.175443+00:00.
Calculation fingerprint: 50e0018b602d1209553ab05b9f985675d61e74d912daa507b54699a579a8be88


In [2]:
# 6. Controls only filter and display the loaded rows.
controls = DurationControls(loaded_results, saved_metadata["configuration"]["historical_events"])
display(controls.ui)
controls.render()